# US Rates Options — Vol Surface PCA

Consolidates the full research: data exploration, pre-modelling prep, book risk, hedging,
and framework robustness.

**Files:** `volsurface.py` (model + analysis) · `volsurface_plots.py` (charts) · this notebook.

**Locked-in decisions** (each justified by a section below):

| Decision | Section |
|---|---|
| bp diffs, not log/relative | §3 |
| drop the 2D expiry row | §1 |
| `weekday_rolling` calendar cleaning | §2 |
| per-window static σ, EWMA rejected | §3 |
| window = 500d, on identification grounds | §5, §9 |
| 6 factors, not 3 | §6, §8 |
| rank factors by book PnL variance, not surface variance | §6 |

**Central identity:** `PnL_t = vega' Δv_t = u' z_t`, `u = σ ⊙ vega`. The book's PnL is a
one-dimensional projection of the standardised surface. A k-factor model explains PnL
**iff** `u` lies in the span of the top-k PCs.

In [ ]:
%matplotlib inline
import warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 200, "display.max_columns", 60)

import volsurface as vs
import volsurface_plots as vp
vp.set_style()

## 0. Data

In [ ]:
import pyjade.jade as jade
from pyjade.pearl import pearl_service
from DataManager import get_vol, get_vega

jade.JadeSession()
pearl_service().ARM_ETCConnect()
pearl_service().ARM_SwitchToETK()

INDEX, CCY, CLOSING = "SOFR", "USD", "NY"
BOOKS = ["swopt_book", "irg_book"]
VOL_START, VOL_END = "01/02/2017", "08/13/2026"

swopt_vols = get_vol(INDEX, CCY, "SWOPT", CLOSING, VOL_START, VOL_END)
irg_vols   = get_vol(INDEX, CCY, "IRR",   CLOSING, VOL_START, VOL_END)
swopt_vega = get_vega(VOL_START, VOL_END, BOOKS, "SWOPT")
irg_vega   = get_vega(VOL_START, VOL_END, BOOKS, "IRR")

- **What:** fetch vols and book vega.
- **Why:** `get_vol`/`get_vega` return `HistoricalSurfaces` ordered **most-recent-first**,
  and vega sits on trade-level pillars, not vol pillars. `vs.prepare_grid` and
  `vs.align_vega` handle the reordering and the vega-conserving rebucketing.
- **Interpret:** confirm the vega date range printed below. It is typically far shorter
  than the vol history — never assume today's book was held back to 2017.

In [ ]:
CFG = vs.PCAConfig(n_components=6, n_report=8, window=500, refit_every=21)

# Check axis labels BEFORE gridding. SWOPT and IRG do not share a tenor axis:
# SWOPT tenor = swap tenor (1Y..30Y); IRG tenor = index tenor (1M/3M/6M/12M).
vs.describe_grid(swopt_vols, "SWOPT"); print()
vs.describe_grid(irg_vols, "IRG"); print()

grid     = vs.prepare_grid(swopt_vols, drop_expiries=CFG.drop_expiries)
raw_ch   = vs.to_changes(grid, "diff")
clean_ch = vs.clean_calendar(raw_ch, method=CFG.calendar_method,
                             window=CFG.calendar_window).dropna(how="any")

vega_panel = vs.align_vega(swopt_vega, grid.columns)          # dates x grid
vega       = vs.align_vega(swopt_vega, grid.columns, aggregate="ewma", verbose=False)

print(f"\nvol history  : {grid.index[0].date()} -> {grid.index[-1].date()} ({len(grid)} days)")
print(f"vega history : {vega_panel.index[0].date()} -> {vega_panel.index[-1].date()} ({len(vega_panel)} days)")
print(vs.scope_check(vega, grid.columns).to_string(index=False))

## 1. Surface structure and data quality

### 1.1 What the surface looks like
- **What:** a snapshot heatmap, the term structure by tenor, individual node histories.
- **Why:** a model built without looking at its inputs gets surprised by them.
- **Interpret:** the surface should decay smoothly along expiry. Kinks at a single pillar
  usually mean an interpolation seam or a stale quote, not a market feature.

In [ ]:
vp.plot_surface_snapshot(grid); plt.show()
vp.plot_node_timeseries(grid, [("1M","2Y"), ("1Y","5Y"), ("5Y","10Y"), ("10Y","30Y")], clean_ch); plt.show()

### 1.2 Per-node statistics and staleness
- **What:** std, kurtosis and lag-1 autocorrelation per node; share of days with an
  exactly-zero change; overlay against where the book actually holds vega.
- **Why:** PCA weights nodes by variance. A stale node contributes fake structure, and a
  near-zero σ becomes a divide-by-noise in standardisation.
- **Interpret:**
  - `zero_share > 0.20` → node not re-marked daily; consider dropping it.
  - `lag1_ac < -0.10` → bid/ask bounce or stale marks, **not** a real factor.
  - The dangerous case is high staleness **where the book has vega** — compare the two
    heatmaps. Staleness away from the book is cosmetic.

In [ ]:
stats = vs.point_stats(clean_ch)
print(stats.sort_values("zero_share", ascending=False).head(10).round(4).to_string())

vp.plot_point_stats(stats); plt.show()
vp.plot_staleness(vs.staleness_map(clean_ch), vega); plt.show()

noise = vs.marking_noise_diagnostics(clean_ch)
print(f"\nnodes flagged as marking-noise suspects: {int(noise['suspect'].sum())} / {len(noise)}")
print(noise[noise["suspect"]].sort_values("lag1_ac").head(10).round(4).to_string())

### 1.3 Grid validation
- **What:** T/p ratio on the chosen grid; which requested pillars are actually present.
- **Why:** a covariance on p points needs T ≫ p. At T < p it is singular and every trailing
  factor is noise.
- **Interpret:** T/p ≥ 10 is comfortable, 3–10 workable, < 3 weak, < 1 unusable. Note the
  ratio that matters for a fit is **window / p**, not full-sample / p.

**2D is dropped** — it duplicates 1M. Removing it *raised* PC1's share (74% → 76% in the
research): in a correlation PCA, removing a low-average-correlation column raises mean
correlation and concentrates PC1, the opposite of the intuition for a duplicate row.

In [ ]:
p = clean_ch.shape[1]
print(f"full sample T/p = {len(clean_ch)}/{p} = {len(clean_ch)/p:.1f}")
print(f"in-window  T/p = {CFG.window}/{p} = {CFG.window/p:.1f}"
      f"{'   <-- WEAK, trailing factors are noise' if CFG.window < 3*p else ''}")

## 2. The calendar / expiry-roll effect

- **What:** mean daily change by weekday, per expiry row.
- **Why:** several consecutive business days share one expiry date, so short-expiry vol
  grinds one way then jumps at the roll. That is a deterministic sawtooth in the *changes*,
  which are the PCA input — it manufactures a spurious factor.
- **Interpret:** a large weekday spread on 1M/3M and near-zero further out is the
  signature. The *shape* is not predictable a priori — the research predicted
  Monday-jump/Tue-Fri-grind and the data showed Mon+/Tue+/Wed−. This is exactly why the
  `theta` fix needs the **real** ARM pillar expiry dates rather than an assumed convention.

In [ ]:
prof = vs.weekday_profile(raw_ch)
print(prof.round(4).to_string())
vp.plot_weekday_profile(prof); plt.show()

### 2.1 Choosing the cleaning method
- **What:** compare `none` vs `weekday_rolling` on weekday leakage, variance retained,
  PC1 share, effective dimensionality, and lag-5 autocorrelation of the short rows.
- **Why:** a cleaning method can remove the artefact, or destroy real signal, or
  manufacture false coherence. All three look similar unless measured.
- **Interpret:** want leakage ↓ with variance retained ≈ 100%. `project` and `overlap`
  are **not implemented** — both were tested on real data and failed (`project` destroyed
  ~20% of variance and *raised* PC1 concentration; `overlap` induced negative lag-5
  autocorrelation).

In [ ]:
print(vs.calendar_scorecard(raw_ch, CFG, methods=("none", "weekday_rolling")).to_string())
vp.plot_calendar_scorecard(vs.calendar_scorecard(raw_ch, CFG)); plt.show()

### 2.2 Acceptance test
- **What:** `corr(PC score, weekday dummy)`, before vs after cleaning.
- **Why:** this is the metric that decides whether the cleaning worked. It is the number
  to quote to the desk.
- **Interpret:** **read the signal PCs only.** Degenerate trailing PCs carrying a few bp of
  variance are arbitrary rotations of the noise subspace; leftover weekday signal
  concentrates there and a naive max over all PCs reports that a *successful* cleaning made
  things worse. Observed directly: the genuine calendar factor improved 0.182 → 0.117 while
  a PC holding 0.06% of variance went 0.181 → 0.233 and hijacked the headline. The verdict
  is `attrs['worst_signal']`; grey bands in the chart are noise PCs.

In [ ]:
m_raw   = vs.VolPCA(config=CFG).fit(raw_ch.dropna(how="any"))
m_clean = vs.VolPCA(config=CFG).fit(clean_ch)
cal_raw   = vs.calendar_acceptance(m_raw.transform(raw_ch.dropna(how="any"), k=8), model=m_raw)
cal_clean = vs.calendar_acceptance(m_clean.transform(clean_ch, k=8), model=m_clean)

print(f"signal-PC worst : {cal_raw.attrs['worst_signal']:.3f} -> {cal_clean.attrs['worst_signal']:.3f}")
print(f"naive all-PC max: {cal_raw.attrs['worst_all']:.3f} -> {cal_clean.attrs['worst_all']:.3f}"
      "   <- noise-dominated, do NOT quote")
vp.plot_calendar_acceptance(cal_raw, cal_clean); plt.show()

## 3. Is this a PCA candidate? Correlation, dimensionality, transforms

### 3.1 Correlation structure
- **What:** correlation matrix of daily changes, distribution of pairwise correlations,
  effective dimensionality `(Σλ)²/Σλ²`.
- **Why:** PCA is worth doing exactly when a large grid behaves like a few factors.
- **Interpret:** effective dimension = 1 means perfectly one-factor, = p means independent.
  A low effective dimension on a large grid is the whole justification for the model.

In [ ]:
summ = vs.correlation_summary(clean_ch)
print({k: round(v, 3) if isinstance(v, float) else v for k, v in summ.items()})
vp.plot_correlation(clean_ch, summ); plt.show()

### 3.2 Flat-vol check on the IRG feed
- **What:** correlation between expiries N pillars apart, IRG vs SWOPT — along the
  **expiry** axis at a fixed tenor, which is where the flat-vol artefact lives.
- **Axis warning:** IRG tenors are **index** tenors (1M/3M/6M/12M), not swap tenors, so
  `prepare_grid` needs `tenors=vs.IRG_TENORS`. Passing the SWOPT grid returns an empty
  frame — that is what broke §3.2/3.3/§10 on the first run.
- **Why:** a *flat* cap vol at expiry T prices every caplet out to T — a cumulative
  average, not a point on a surface. Adjacent expiries then share most of their
  optionality by construction, and a PCA returns a ~99% PC1 that is a quoting artefact.
- **Interpret:** SWOPT is the honest baseline (genuine point quotes). If IRG sits far above
  it and decays much more slowly, **the feed is cumulative and must be stripped to caplet
  vols before any of this analysis is valid.**

In [ ]:
grid_irg = vs.prepare_grid(irg_vols, expiries=vs.IRG_EXPIRIES, tenors=vs.IRG_TENORS,
                           drop_expiries=CFG.drop_expiries)
ch_irg   = vs.clean_calendar(vs.to_changes(grid_irg), verbose=False).dropna(how="any")

IRG_TEN, SW_TEN = "3M", "10Y"          # each at its own most-liquid tenor
curves = {f"SWOPT @{SW_TEN}": vs.neighbour_correlation(clean_ch, SW_TEN),
          f"IRG @{IRG_TEN}":  vs.neighbour_correlation(ch_irg, IRG_TEN)}
print(pd.DataFrame(curves).round(3).to_string())
vp.plot_neighbour_correlation(curves); plt.show()

gap = (curves[f"IRG @{IRG_TEN}"] - curves[f"SWOPT @{SW_TEN}"]).mean()
print(f"\nmean gap = {gap:.3f}  ->  "
      f"{'WARNING: IRG looks CUMULATIVE — strip to caplet vols' if gap > 0.10 else 'IRG looks like point quotes'}")

### 3.3 The seam between SWOPT and IRG
- **Corrected premise.** The original worry was a duplicated 12M/1Y column. That was
  wrong: IRG's 12M is an *index* tenor and SWOPT's 1Y is a *swap* tenor — different
  objects, not one series arriving twice. `seam_test` correctly returns nothing.
- **The real issue is the rebucketing.** `bucket_matrix` clamps every source tenor at or
  below the first target pillar into that pillar, and every IRG index tenor is ≤ 1Y — so
  **100% of IRG vega collapses into the 1Y swap-tenor column**.
  `DataManager.vega_to_x_format` does exactly this, so it is the desk's existing
  convention, but it destroys the cap tenor axis.
- **Consequence:** IRG must be carried as its own block (§10), never merged into SWOPT.

In [ ]:
seam = vs.seam_test(clean_ch, ch_irg)
print("shared pillars:", len(seam), "(expected 0 — different axes)\n")
vs.irg_tenor_collapse_check(vs.IRG_TENORS)

### 3.4 Transform choice and normality
- **What:** bp diffs / relative / log, each raw, z-scored, or EWMA-scaled, scored on
  marginal kurtosis and the joint χ² QQ slope.
- **Why:** the trader asked for standardisation; the question is *which* transform gets
  closest to multivariate normal, since every risk number assumes it.
- **Interpret:** χ² slope 1.0 = multivariate normal.
  - **`raw` and `zscore` rows are identical by construction.** Mahalanobis distance is
    affine-invariant, so a constant per-node σ *cannot* change the joint fit; only a
    time-varying scaling (`ewma`) can. If they differ, the test is broken.
  - **On this data bp diffs are the WORST transform on both metrics** (highest kurtosis,
    χ² slope furthest from 1); log is the best. All slopes sit ~4.3–4.9 — nothing here is
    close to multivariate normal, tails dominate.
  - **bp diffs remain the right choice, but for a different reason than normality:**
    PnL = vegaᵀΔv is linear in bp changes. Log changes break that identity and would
    require rewriting the whole risk/hedge stack for a modest gain in an assumption the
    data violates badly anyway. Record the cost: **λₖ understates tail risk** — scale VaR
    rather than trusting Gaussian quantiles.

In [ ]:
tc = vs.transform_comparison(grid, CFG)
print(tc.sort_values("chi2_dist_from_1").to_string())
vp.plot_transform_comparison(tc); plt.show()
vp.plot_normality(clean_ch, [("1M","2Y"), ("1Y","5Y"), ("5Y","10Y"), ("10Y","30Y")]); plt.show()

## 4. Regimes

- **What:** per-regime vol level, correlation and effective dimension; pairwise subspace
  overlap between regimes **against a split-half noise floor**.
- **Why:** if the factor structure genuinely differs by regime, one all-weather model is
  wrong. If it doesn't, regime-conditional loadings add parameters for nothing.
- **Interpret:** a raw overlap of 0.85 means nothing until you know what two random halves
  of the *same* regime score. Overlaps **inside** the floor are sampling noise, not a
  regime change. Bold entries in the chart are genuinely below the floor.
  - **On this data several pairs sit genuinely BELOW their floors** — Autumn 2023 vs
    Covid 2020 / Calm 2021 / SVB 2023 / Tariffs 2025, and Covid 2020 vs Tariffs 2025. So
    the structure is *not* uniformly stable across regimes and Autumn 2023 is the clear
    outlier. Check that period's dates and marks before calling it a market fact.
  - Regime dates live in `vs.REGIMES` — edit them to the desk's own view before quoting.

In [ ]:
print(vs.regime_stats(clean_ch).round(3).to_string())
vp.plot_regime_stats(vs.regime_stats(clean_ch)); plt.show()

In [ ]:
M = vs.regime_comparison(clean_ch, CFG, k=6)
print(M.to_string())
print("\nsplit-half noise floors:"); print(M.attrs["noise_floor"].round(3).to_string())
vp.plot_regime_overlap(M); plt.show()

## 5. The PCA fit

### 5.1 Baseline
- **What:** eigenvalue spectrum against the Marchenko–Pastur noise edge; loadings as
  expiry × tenor heatmaps.
- **Why:** MP gives a principled answer to "how many factors are real" — eigenvalues inside
  the bulk are indistinguishable from i.i.d. noise.
- **Interpret:**
  - Loadings should read as level (PC1, all one sign), term-structure slope (PC2, sign
    change along expiry), twist (PC3). A factor that is **one hot cell** is a stale or
    mismarked node, not a market factor.
  - The cumulative-variance panel is deliberately labelled as *not* the metric that
    matters — see §6.

In [ ]:
model = vs.VolPCA(config=CFG).fit(clean_ch)
print(model); print()
print(model.summary().to_string())
vp.plot_scree(model); plt.show()
vp.plot_loadings(model, k=4); plt.show()

### 5.1b Export PC loadings as surfaces (trader-sheet layout)

- **What:** the same loadings as the heatmaps above, reshaped to expiry × tenor blocks
  on the trader's 16 × 11 grid and written to CSV in his stacked `#1 / #2 / #3 ...`
  layout.
- **Why:** the sheet is the only common reference between this model and the incumbent.
  Until the two are on the same grid, in the same units, with the same sign convention,
  "our PC2" and "his #2" are not comparable and no disagreement can be attributed.
- **Interpret:**
  - **Units.** The sheet is in **bp of vol per +1σ shock of the factor**
    (`σ ⊙ L_j · √λ_j`), not unit-norm eigenvector entries. His #1 runs 4.77 at 1Mx1Y to
    0.36 at 30Yx30Y, which back out to daily σ of 5.4 and 0.41 bp — the give-away.
    `mode='unit'` returns the raw eigenvectors if you want to compare shapes instead.
  - **Grid.** His grid is 16 × 11 = 176 nodes; ours is 13 × 10 = 130. Expiries
    **12Y / 25Y / 30Y** and tenor **25Y** are not on the SOFR SWOPT feed, so 46 cells
    per block come back blank. They are populated on his sheet, which means it is built
    on an interpolated or extrapolated grid — worth asking him which.
  - **9M.** His sheet does not carry a 9M expiry either. That settles the open question
    from the model spec: excluding 9M matches the desk, keep it out.
  - **Sign.** We anchor on `positive_sum`. If a PC comes out mirrored against his, flip
    it — the sign of an eigenvector is arbitrary and flipping changes nothing downstream.


In [ ]:
# --- 5.1b  PC loadings as surfaces, trader-sheet layout ---------------------
# Sheet geometry read off the Risk tab: "#1" header on row 4, "#2" on row 22,
# "#3" on row 40 -> 18-row pitch = 1 header + 16 expiry rows + 1 Total row.
SHEET_EXPIRIES = ["1M", "3M", "6M", "1Y", "18M", "2Y", "3Y", "4Y", "5Y", "7Y",
                  "10Y", "12Y", "15Y", "20Y", "25Y", "30Y"]
SHEET_TENORS = ["1Y", "2Y", "3Y", "4Y", "5Y", "7Y", "10Y", "15Y", "20Y", "25Y", "30Y"]


def pc_surfaces(model, k=8, mode="bp_1sd",
                expiries=SHEET_EXPIRIES, tenors=SHEET_TENORS):
    """PC loadings reshaped to expiry x tenor surfaces on the trader's grid.

    mode:
      'bp_1sd'  sigma * L_j * sqrt(lam_j)  -- bp of vol per +1sd shock of factor j.
                This is the sheet's convention.
      'z_1sd'   L_j * sqrt(lam_j)          -- same, in z units (no sigma).
      'unit'    L_j                        -- the raw unit-norm eigenvector.

    Nodes absent from the model grid come back NaN so the block still lines up
    row-for-row with the sheet.
    """
    if model.loadings is None:
        raise ValueError("model is unfitted")
    k = min(k, model.loadings.shape[1])
    L = model.loadings.iloc[:, :k]

    if mode == "unit":
        S = L
    elif mode == "z_1sd":
        S = L.mul(np.sqrt(model.eigenvalues.iloc[:k]), axis=1)
    elif mode == "bp_1sd":
        S = (L.mul(np.sqrt(model.eigenvalues.iloc[:k]), axis=1)
              .mul(model.scale.sigma.reindex(L.index), axis=0))
    else:
        raise ValueError(f"unknown mode {mode!r} — use 'bp_1sd', 'z_1sd' or 'unit'")

    out = {}
    for j, pc in enumerate(S.columns, start=1):
        m = S[pc].unstack("tenor").reindex(index=expiries, columns=tenors)
        m.index.name, m.columns.name = "expiry", "tenor"
        out[f"#{j}"] = m
    return out


def write_pc_sheet(surfs, path, decimals=2, total_row=True):
    """Stack the surfaces into the sheet's vertical block layout and write CSV.

    Block = header row (label + tenors), one row per expiry, then Total.
    Paste the CSV's top-left cell onto B4 of the Risk tab and every block lands
    on the same row as his. Set total_row=False if row 21 is blank on his sheet
    rather than a total.
    """
    rows = []
    for name, m in surfs.items():
        rows.append([name] + list(m.columns))
        for e in m.index:
            rows.append([e] + ["" if pd.isna(v) else round(float(v), decimals)
                               for v in m.loc[e].values])
        if total_row:
            rows.append(["Total"] + ["" if m[t].isna().all()
                                     else round(float(m[t].sum()), decimals)
                                     for t in m.columns])
    width = max(len(r) for r in rows)
    df = pd.DataFrame([r + [""] * (width - len(r)) for r in rows])
    df.to_csv(path, index=False, header=False)
    return df


K_EXPORT = 8
surfs = pc_surfaces(model, k=K_EXPORT, mode="bp_1sd")
sheet = write_pc_sheet(surfs, "pc_loadings_swopt.csv")

print(f"fit window   : {model.fit_index[0].date()} -> {model.fit_index[-1].date()} "
      f"({len(model.fit_index)} obs, p={len(model.grid)})")
print(f"units        : bp of vol per +1sd factor shock  (sigma * L * sqrt(lambda))")
print(f"exported     : {K_EXPORT} PCs, {sheet.shape[0]} rows x {sheet.shape[1]} cols "
      f"-> pc_loadings_swopt.csv")

miss_e = [e for e in SHEET_EXPIRIES if e not in set(model.grid.get_level_values("expiry"))]
miss_t = [t for t in SHEET_TENORS if t not in set(model.grid.get_level_values("tenor"))]
print(f"blank on ours: expiries {miss_e} | tenors {miss_t} "
      f"-> {int(surfs['#1'].isna().sum().sum())} of {surfs['#1'].size} cells per block")

chk = pd.DataFrame({
    "surface_%": (100 * model.explained.iloc[:K_EXPORT]).round(2).values,
    "sqrt_lam": np.sqrt(model.eigenvalues.iloc[:K_EXPORT]).round(3).values,
    "max_abs_bp": [float(surfs[f"#{j}"].abs().max().max()) for j in range(1, K_EXPORT + 1)],
    "loading_sum": [float(model.loadings.iloc[:, j - 1].sum()) for j in range(1, K_EXPORT + 1)],
}, index=[f"#{j}" for j in range(1, K_EXPORT + 1)]).round(3)
print("\nsign anchor is positive_sum — flip any PC that comes out mirrored vs his sheet")
print(chk.to_string())

for j in (1, 2, 3):
    print(f"\n--- PC #{j}  (bp per +1sd) " + "-" * 40)
    print(surfs[f"#{j}"].round(2).to_string(na_rep=""))


### 5.2 Eigenvalue decay vs estimation window
- **What:** decay profile and count of factors above the MP edge, per window.
- **Why:** separates a property of the market (how concentrated the surface is) from a
  property of the sample (how many factors you can *resolve*).
- **Interpret:** `n_above_MP` rises with the window because the noise bulk narrows as T
  grows. A short window reporting 2 significant factors is telling you about sample size,
  **not** that the market got simpler. On the log panel a straight line means geometric decay.

In [ ]:
WINDOWS = [125, 250, 375, 500, 750, 1000]
decay = vs.eigen_decay_table(clean_ch, WINDOWS, k=12)
print(decay[[f"PC{i}" for i in range(1,7)] + ["n_above_MP","MP_edge","T/p","eff_dim"]].to_string())
vp.plot_eigen_decay(decay, k=12); plt.show()

### 5.3 Are the scores well-behaved?
- **What:** QQ plots and autocorrelation of the leading PC scores.
- **Why:** PCA does not *need* i.i.d. Gaussian scores to be a valid rotation, but every risk
  number built on λ<sub>k</sub> does.
- **Interpret:**
  - negative `lag1_ac` → stale marks or bid/ask bounce leaking into a factor;
  - `abs_ac1` (vol clustering) is almost always present — that is *why* σ must be
    re-estimated at each refit;
  - excess kurtosis → λ<sub>k</sub> understates tail risk; scale VaR accordingly.

In [ ]:
scores = model.transform(clean_ch, k=8)
diag = vs.score_diagnostics(scores)
print(diag.to_string())
vp.plot_score_diagnostics(scores, diag, k=4); plt.show()

## 6. Book risk — the headline

- **What:** rank factors by **share of book PnL variance** alongside surface variance;
  decompose the book direction `u` in the PC basis.
- **Why:** `PnL = u'z` is one-dimensional. A k-factor model explains PnL **iff** `u` lies
  in span(top-k). PCA maximises *surface* variance and has no reason to align with `u`.
- **Interpret:**
  - **Lead with this table, not the scree plot.** In the research the two rankings came out
    nearly opposite: PC1 was 76% of surface variance and ~0.7% of book risk, while PC3+PC5
    were 5.6% of surface variance and 67% of book risk.
  - `exposure_$` is dollar PnL per +1σ move of that factor — the number you hedge.
  - `pnl_r2_implied` is the analytic version of the walk-forward backtest in §8. If they
    disagree, one of them is wrong.
  - **Caveat:** the book's near-zero PC1 exposure is very likely the *output of the
    incumbent PCA hedge*, not an independent market observation. Treat it as a diagnostic
    clue about their model, not a property of the market.

In [ ]:
risk  = vs.pc_risk_exposure(model, vega)
align = vs.book_direction_alignment(model, vega)
print(risk[["exposure_$","pnl_var_%","surface_var_%","risk_rank","surface_rank"]].to_string())
print(f"\ndaily book PnL sd from the factor model: {risk.attrs['daily_pnl_sd']:,.0f}")
vp.plot_risk_ranking(risk); plt.show()
vp.plot_book_alignment(align); plt.show()

### 6.1 Holding horizon
- **What:** PnL R² at 1, 5, 10, 21-day horizons.
- **Why:** if the 1-day residual is marking noise rather than risk, it washes out when
  aggregated.
- **Interpret:** R² rising with horizon → the daily residual is noise, and the model is
  better than the 1-day number suggests. Flat → the residual is genuine unmodelled risk.

In [ ]:
h = vs.horizon_comparison(clean_ch, vega, CFG)
print(h.to_string())
vp.plot_horizon(h); plt.show()

## 7. Factor identification and stability

- **What:** two different things, and conflating them is a real error.
  - **subspace stability** — does the *span* of the top-k factors persist? Robust,
    relabelling-invariant.
  - **per-PC identification** — does "PC3" mean the same direction after each refit? This
    is what a **named hedge** depends on.
- **Why:** when two eigenvalues are close their order is unstable, so PC3 and PC4 trade
  places and every hedge ratio labelled "PC3" silently changes meaning.
- **Interpret:** `swap_rate > 0.10` or `mean_overlap < 0.90` → you cannot put a named hedge
  on that factor. Reported **per PC**, never aggregated: an aggregate is dominated by
  degenerate trailing PCs and gives a backwards answer.

**This was the load-bearing diagnostic for the incumbent hypothesis** — a factor that is
unstable *and* carries most of the book's risk produces exactly the reported symptom.
**But on this data no PC is unidentified at any window** (max swap rate 0.074 at W=125,
overlaps ≥ 0.83). Factor churn is therefore *not* the explanation for the incumbent's
large errors, and that hypothesis should be dropped unless their window is far shorter
than 125d. Look instead at hedge churn (§8.1) and at what sits beyond PC k (§8).

In [ ]:
rows = []
for w in [125, 250, 500, 750]:
    try:
        st = vs.pc_stability_across_refits(clean_ch, CFG.copy(window=w), k=6, verbose=False)
        for pc, r in st.iterrows():
            rows.append({"window": w, "pc": pc, **r.to_dict()})
    except ValueError as e:
        print(f"window {w}: {e}")
stab_pc = pd.DataFrame(rows)
print("swap rate:"); print(stab_pc.pivot(index="pc", columns="window", values="swap_rate").to_string())
print("\nmean overlap:"); print(stab_pc.pivot(index="pc", columns="window", values="mean_overlap").to_string())
vp.plot_identification(stab_pc); plt.show()

### 7.1 Is the structure stationary?
- **What:** rolling PC variance shares with regime shading; the same PC's loading profile
  at several refit dates.
- **Why:** a model fitted in a calm window will misprice risk in a stressed one if the
  structure moves.
- **Interpret:** a **rising PC1 share is the classic stress signature** — the surface
  collapses toward a single factor exactly when it matters. In the loading-drift panels,
  lines that stay on top of each other = a stable, hedgeable factor.

In [ ]:
share = vs.rolling_factor_share(clean_ch, CFG, k=5, step=10)
vp.plot_rolling_share(share, vs.REGIMES); plt.show()
print(share.describe().round(2).to_string())

In [ ]:
step = max(1, (len(clean_ch) - CFG.window) // 5)
snaps, prev = {}, None
for s in range(0, len(clean_ch) - CFG.window + 1, step):
    prev = vs.VolPCA(config=CFG).fit(clean_ch.iloc[s:s+CFG.window], reference=prev)
    snaps[str(clean_ch.index[s+CFG.window-1].date())] = prev
vp.plot_loading_drift(snaps); plt.show()
vp.plot_subspace_matrix(snaps, k=6, title="Subspace overlap between refit dates"); plt.show()

## 8. Hedging

- **What:** greedily select instruments that make the exposure matrix `A` well-conditioned,
  then solve `A h = -b` for notionals.
- **Why:** conditioning is not cosmetic. In the research a greedy pick gave κ = 10.8 vs
  159.9 for a naive liquidity-driven pick — a 15× difference in how much hedge notionals
  amplify estimation error, from instrument choice alone.
- **Interpret:**
  - `min_singular` small → the candidate set contains no distinct direction for one factor,
    and the solver answers with huge offsetting notionals. **Widen the candidate set**
    rather than accepting it.
  - `gross_ratio` much above ~1.5× deserves scrutiny before it goes to the desk.
  - `unhedged_var_%` is the honest number: book PnL variance sitting **beyond** PC k that
    this hedge cannot touch.
  - Long-expiry instruments have small σ, so they carry little factor exposure per unit of
    vega; the solver keeps wanting short-expiry instruments, which are mutually collinear.

In [ ]:
cands = vs.unit_vega_instruments(model.grid)
sel   = vs.select_hedge_instruments(model, cands, k=CFG.n_components)
sol   = vs.solve_hedge(model, vega, cands[sel["instruments"]], k=CFG.n_components)
print("\nselected:", [f"{a}x{b}" for a, b in sel["instruments"]])
vp.plot_hedge_report(sel, sol); plt.show()

### 8.1 Hedge churn
- **What:** re-solve at every refit on a fixed instrument set; track notionals, turnover
  and conditioning.
- **Why:** a hedge that solves perfectly but churns every refit is a transaction-cost
  problem masquerading as a risk model.
- **Interpret:** turnover spikes should line up with the PC swap events in §7. If they do,
  that is the incumbent failure mechanism demonstrated rather than argued.

In [ ]:
fixed = cands[sel["instruments"]]
hist, kap, prev = [], [], None
for s in range(0, len(clean_ch) - CFG.window + 1, CFG.refit_every):
    prev = m = vs.VolPCA(config=CFG).fit(clean_ch.iloc[s:s+CFG.window], reference=prev)
    hist.append(vs.solve_hedge(m, vega, fixed, k=CFG.n_components, verbose=False)["notionals"])
    kap.append(np.linalg.cond(vs.hedge_exposure_matrix(m, fixed).values))
H = pd.DataFrame(hist, index=clean_ch.index[CFG.window-1::CFG.refit_every][:len(hist)])
vp.plot_hedge_churn(H, kap); plt.show()

## 9. Walk-forward PnL explain and window robustness

### 9.1 Backtest
- **What:** strictly out-of-sample — loadings, μ and σ frozen before the days they score.
- **Why:** in-sample variance explained mechanically improves with more factors and shorter
  windows. Only a walk-forward answers the real question.
- **Interpret:**
  - Compare against the analytic `pnl_r2_implied` from §6 — they should agree.
  - A **negative** R² at k=1 means using the dominant factor as a PnL model is worse than
    predicting zero: a small net exposure multiplied by large PC1 moves generates noise.
  - Check whether the worst days cluster near refit dates (factor churn) or near regime
    turns (stale covariance).

In [ ]:
bt = vs.pnl_explain_backtest(clean_ch, vega, CFG, k_list=(1,2,3,4,5,6,8,10))
vp.plot_backtest(bt, vs.REGIMES); plt.show()

comp = pd.DataFrame({"walk_forward": bt["r2"],
                     "analytic": [align["pnl_r2_implied"].iloc[k-1] if k <= len(align) else np.nan
                                  for k in bt["r2"].index]})
comp["gap"] = comp["walk_forward"] - comp["analytic"]
print(comp.round(4).to_string())

### 9.2 Does window W forecast *future* covariance?
- **What:** estimate covariance on the trailing W days, score it on the next 21 days it has
  never seen. Compare the k-factor covariance against the **raw sample covariance**.
- **Why:** this is the point of the whole exercise — historical covariance is only useful if
  it forecasts future covariance.
- **Interpret:**
  - **`mvp_oos_vol`** — realized OOS vol of the minimum-variance portfolio built from the
    estimate. **Lower is better.** The most demanding test available: MVP weights load
    precisely on the smallest, worst-estimated eigen-directions, so it punishes exactly the
    errors a Frobenius norm hides behind the large diagonal.
  - **`vol_underestimate_x` > 1** → the model promised less risk than it delivered. Where
    W < p the *sample* covariance is singular, the optimiser finds a phantom zero-variance
    direction, and this ratio explodes. That gap is the practical argument for a factor
    model over a raw covariance.
  - **`calib_slope`** — 1.0 is perfect; < 1 under-predicts risk.
  - `frob_rel_err` is a *ranking* only — realized covariance over 21 days is itself noisy.
  - **`factor_beats_sample` can be False, and on this data it is at W ≥ 375.** With
    T/p ≈ 18.6 the sample covariance is well estimated at long windows, and truncating to
    k factors discards more real structure than noise. The factor model only wins where
    data is scarce (W = 125, 250). That argues for the factor model as a *risk attribution
    and hedging* basis, not as the best covariance forecaster.

In [ ]:
cov = vs.covariance_forecast_test(clean_ch, WINDOWS, k_list=(3,5,6,10), horizon=21, step=21)
print(cov.pivot(index="window", columns="estimator", values="mvp_oos_vol").to_string())
vp.plot_covariance_forecast(cov); plt.show()

### 9.3 Stability vs the split-half noise floor
- **What:** subspace overlap between consecutive refits, as a function of window, against
  the overlap two random halves of the *same* sample produce.
- **Why:** short windows chase noise; long windows are stable but stale. This is the
  estimation-noise half of the bias/variance picture — §9.2 carries the bias half.
- **Interpret:** a window whose overlap sits **at or below the floor** is fitting noise, and
  no downstream care rescues it. That is a principled cut, not a judgement call.

In [ ]:
stab  = vs.window_stability_curve(clean_ch, WINDOWS, k=6, step=21)
floor = vs.split_half_overlap(clean_ch, CFG, k=6, n_draws=25)
print(stab.to_string())
print(f"\nsplit-half floor (k=6): {floor['mean']:.4f}  [p05 {floor['p05']:.4f}, p95 {floor['p95']:.4f}]")
vp.plot_window_robustness(stab, cov, floor=floor, k=6); plt.show()

models_by_w = {f"W={w}": vs.VolPCA(config=CFG.copy(window=w)).fit(clean_ch) for w in WINDOWS}
vp.plot_subspace_matrix(models_by_w, k=6, title="Subspace overlap between windows"); plt.show()

### 9.4 Scorecard
- **Interpret:** the three criteria — covariance forecast, stability, book PnL R² — **do
  not have to agree**, and when they don't, that disagreement *is* the finding. Weight
  identification (§7) most heavily: an unidentified factor makes a named hedge meaningless
  no matter how well the covariance forecasts.

In [ ]:
score = pd.DataFrame(index=sorted(set(WINDOWS) & set(stab["window"])))
score["subspace_overlap"]   = stab.set_index("window")["subspace_overlap"]
score["above_noise_floor"]  = score["subspace_overlap"] > floor["p95"]
score["n_above_MP"]         = decay["n_above_MP"]
score["oos_mvp_k6"]         = cov[cov.estimator=="k=6"].set_index("window")["mvp_oos_vol"]
score["oos_mvp_sample"]     = cov[cov.estimator=="sample"].set_index("window")["mvp_oos_vol"]
score["factor_beats_sample"] = score["oos_mvp_k6"] < score["oos_mvp_sample"]
score["calib_k6"]           = cov[cov.estimator=="k=6"].set_index("window")["calib_slope"]
print(score.round(4).to_string())

print(f"\nbest by covariance forecast : {score['oos_mvp_k6'].idxmin()}")
print(f"best by subspace stability  : {score['subspace_overlap'].idxmax()}")
unident = stab_pc[stab_pc.swap_rate > 0.10].groupby("window")["pc"].apply(list)
print(f"PCs NOT identified by window: {dict(unident) if len(unident) else 'none at any window'}")

## 10. IRG: the two-stage hierarchical model

- **What:** fit SWOPT factors first, regress IRG on the SWOPT scores, then extract basis
  factors from the IRG residual.
- **Why:** a naive blend lets the noisier IRG block distort the SWOPT factors, and the
  shared-pillar seam (§3.3) creates near-collinear columns. Two-stage keeps the SWOPT
  factors clean and carries IRG's incremental risk explicitly as a basis factor.
- **Interpret:** `irg_explained_by_swopt` high → IRG is nearly redundant and the basis
  factor is small. Low → there is genuine cap/floor-specific risk that a SWOPT-only model
  cannot see, and PC2 of the basis must be carried explicitly.
- **Prerequisite:** settle §3.2 first. If IRG is quoted flat (cumulative) this model
  inherits the problem.
- **The two blocks have different tenor axes** — intended. The regression is on the SWOPT
  *scores* (a time series), not on matching columns.

In [ ]:
common = clean_ch.index.intersection(ch_irg.index)
ts = vs.two_stage_model(clean_ch.loc[common], ch_irg.loc[common], CFG, k_basis=2)
print(f"\nIRG variance explained by SWOPT factors: {100*ts['irg_explained_by_swopt']:.1f}%")
print(ts["basis_model"].summary(k=4).to_string())
vp.plot_loadings(ts["basis_model"], k=2); plt.show()

vega_irg = vs.align_vega(irg_vega, grid_irg.columns, aggregate="ewma", verbose=False)
print(vs.two_stage_risk_split(ts, vega, vega_irg).to_string(index=False))

## 11. The trader's weighted model: beta + time-weighted vega

- **What:** weight the surface by beta to a benchmark node (1Y×10Y) and by time-weighted
  vega, with one parameter λ blending the two.
- **Why it fits here without special-casing:** an importance-weighted PCA is exactly a PCA
  with a different scale vector — `z = (Δv−μ)/(σ/w)`. So it is a different `ScaleModel`, and
  every routine above (risk, hedging, backtest, robustness) runs on it unchanged. That makes
  the comparison genuinely like-for-like.
- **Trade-off:**
  - *buys:* plain PCA maximises surface variance, which is not what the book cares about.
    Up-weighting where the book holds vega tilts the factors toward `u`, raising PnL R² at
    low k.
  - *costs:* the basis stops being a property of the market alone. It moves when the book
    moves, so "PC2" is not comparable across time or across books, the level/slope/twist
    reading weakens, and it cannot be shared with another desk.

> **⚠️ Confirm with the trader: "time-weighted vega" has two standard readings.**
> `mode="ewma"` = time-decayed average of *historical* vega (time = recency), tilting toward
> where the book has been. `mode="expiry"` = vega × T^0.5, the sqrt(T) convention (time =
> time to expiry), tilting long regardless of position. Both are implemented; everything
> below uses `ewma`. Flip the flag and re-run if the other is meant.

In [ ]:
BENCH = ("1Y", "10Y")
wdf = vs.trader_weights(clean_ch, swopt_vega, benchmark=BENCH, beta_window=90,
                        lam=0.5, tw_mode="ewma", halflife=63, cap=10.0)
print(wdf[["beta","tw_vega","weight"]].describe().round(3).to_string())
print(f"\ncorr(beta, |tw_vega|) = {wdf['beta'].corr(wdf['tw_vega'].abs()):.3f}"
      "   (near zero => the two weights carry independent information, so lambda matters)")
vp.plot_weight_diagnostics(wdf, benchmark=BENCH); plt.show()

### 11.1 λ sweep
- **Interpret:** λ = 0 is pure book weighting, λ = 1 is pure market-structure weighting.
  Watch whether weighting raises R² *at low k* (front-loading the book's risk into fewer
  factors) rather than raising the k=6 number — the latter is what matters if the hedge is
  three instruments.

In [ ]:
LAMS = [0.0, 0.25, 0.5, 0.75, 1.0]
sweep, tmodels = {}, {}
for lam in LAMS:
    wl = vs.trader_weights(clean_ch, swopt_vega, benchmark=BENCH, lam=lam,
                           tw_mode="ewma", cap=10.0, verbose=False)["weight"]
    m = vs.fit_weighted_model(clean_ch, wl, CFG, label=f"lam={lam}")
    tmodels[f"lam={lam}"] = m
    al = vs.book_direction_alignment(m, vega, k=8)
    sweep[lam] = {"PC1_surface_%": 100*m.explained.iloc[0],
                  "top6_surface_%": 100*m.explained.iloc[:6].sum(),
                  "PnL_R2_k1": al["pnl_r2_implied"].iloc[0],
                  "PnL_R2_k3": al["pnl_r2_implied"].iloc[2],
                  "PnL_R2_k6": al["pnl_r2_implied"].iloc[5]}
sweep = pd.DataFrame(sweep).T; sweep.index.name = "lambda"
print(sweep.round(4).to_string())

base = {f"PnL_R2_k{k}": float(align["pnl_r2_implied"].iloc[k-1]) for k in (1,3,6)}
print(f"\nUNWEIGHTED benchmark: {({k: round(v,4) for k,v in base.items()})}")
vp.plot_lambda_sweep(sweep, baseline=base); plt.show()

### 11.2 Does the weighted basis clear the same bars?
- **What:** subspace stability of the weighted model vs the unweighted one, weights
  re-estimated at each refit.
- **Why:** a weighted model that explains more PnL but whose factors churn has traded a
  measurable weakness for an unmeasurable one.
- **Interpret:** the weighted basis has **two** sources of drift (the data *and* the
  weights). If its overlap is materially below the unweighted model's, the PnL gain is not
  free.

In [ ]:
cmp = vs.compare_models({"unweighted": model, **{k: tmodels[k] for k in
                        ["lam=0.0","lam=0.5","lam=1.0"]}}, vega)
print(cmp.to_string())
vp.plot_model_comparison(cmp); plt.show()

In [ ]:
prev_u = prev_w = None; ov_u, ov_w, dts = [], [], []
for s in range(0, len(clean_ch) - CFG.window + 1, CFG.refit_every):
    sl = clean_ch.iloc[s:s+CFG.window]
    mu_ = vs.VolPCA(config=CFG).fit(sl)
    try:
        wl = vs.trader_weights(sl, swopt_vega, benchmark=BENCH, lam=0.5, cap=10.0,
                               asof=sl.index[-1], verbose=False)["weight"]
    except (ValueError, KeyError):
        wl = wdf["weight"]
    mw_ = vs.fit_weighted_model(sl, wl, CFG)
    if prev_u is not None:
        ov_u.append(vs.subspace_overlap(prev_u.loadings, mu_.loadings, k=6))
        ov_w.append(vs.subspace_overlap(prev_w.loadings, mw_.loadings, k=6))
        dts.append(sl.index[-1])
    prev_u, prev_w = mu_, mw_

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(dts, ov_u, "o-", ms=3, color=vp.BLUE, label=f"unweighted (mean {np.mean(ov_u):.4f})")
ax.plot(dts, ov_w, "s-", ms=3, color=vp.RED,  label=f"trader lam=0.5 (mean {np.mean(ov_w):.4f})")
ax.axhline(floor["mean"], color="k", ls="--", lw=1, label=f"split-half floor {floor['mean']:.3f}")
ax.set_ylabel("subspace overlap vs prior refit"); ax.legend()
ax.set_title("Does the weighted basis churn more?")
plt.tight_layout(); plt.show()

print("weighted basis drifts MORE" if np.mean(ov_w) < np.mean(ov_u)
      else "weighted basis is no less stable")

## 12. Open items

1. **`theta` calendar cleaning** — implemented but needs the true ARM pillar expiry dates.
   `weekday_rolling` is the interim choice. §2 showed the weekday *shape* is not
   predictable a priori, which is why an assumed roll convention is not good enough.
2. **IRG 12M/1Y seam** — escalate to the curve owner (§3.3). Until settled, treat IRG and
   SWOPT separately and do not trust rebucketed combined vega.
3. **Incumbent comparison** — the missing piece. Need their historical loadings (or realized
   hedge ratios), refit frequency and window, and a list of their large-error dates. Then
   measure subspace overlap between their factors and these over time, and cross-reference
   their error days against the swap events in §7 and the regime turns in §4. That turns
   "here is a new model" into "here is why the old one broke, and here is the fix".
4. **Hedge candidate set** — widen it if §8 shows a small `min_singular`; the current set
   may not contain a genuine third direction.
5. **Time-weighted vega definition** — confirm which reading the trader means (§11).